# ASOS Dataset Benchmark (EarlySign V1)

This notebook demonstrates how to use the V1 framework for both live experimental orchestration and historical backtesting using the ASOS Online-Controlled-Experiment dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import ibis
import pandas as pd

from earlysign.core.ledger import Ledger
from earlysign.schema.ES3.Binomial import ArmData
from earlysign.v1.methods.group_sequential.plan.protocol_design import ProtocolDesigner
from earlysign.v1.templates.binomial_ab import BinomialABTemplate

## 1. Load and Prepare ASOS Data

We use a specific experiment (`3b4300`) and metric from the dataset.

In [ ]:
data_path = "data/asos_digital_experiments_dataset.parquet"
if not os.path.exists(data_path):
    !mkdir -p data
    !wget -O {data_path} https://osf.io/62t7f/download

df_raw = pd.read_parquet(data_path)
exp_id = "3b4300"
df = df_raw[
    (df_raw["experiment_id"] == exp_id)
    & (df_raw["metric_id"] == 1)
    & (df_raw["variant_id"] == 1)
].copy()


def get_increments(df: pd.DataFrame):
    df = df.sort_values("time_since_start")
    sc = df["count_c"] * df["mean_c"]
    dn_c = df["count_c"].diff().fillna(df["count_c"]).astype(int)
    ds_c = sc.diff().fillna(sc).astype(int)
    st = df["count_t"] * df["mean_t"]
    dn_t = df["count_t"].diff().fillna(df["count_t"]).astype(int)
    ds_t = st.diff().fillna(st).astype(int)
    return dn_c, ds_c, dn_t, ds_t


dn_c, ds_c, dn_t, ds_t = get_increments(df)
df["dn_c"], df["ds_c"], df["dn_t"], df["ds_t"] = dn_c, ds_c, dn_t, ds_t
df = df[(df["dn_c"] > 0) | (df["dn_t"] > 0)].copy()
df.head()

## 2. Usage in Practice: Experimental Orchestration

Demonstrating live orchestration with on-demand `progress_report()` calls and the unified `set_protocol` API.

In [ ]:
from earlysign.v1.templates.binomial_ab import BinomialABProtocol

con = ibis.duckdb.connect(":memory:")
ledger = Ledger(con, "live_events")
ledger.ensure()

p_control = df.iloc[0]["mean_c"]
planner = ProtocolDesigner.from_dict({"model": "canonical_joint"})
protocol = planner.plan_binomial_ab(
    alpha=0.05,
    power=0.8,
    p_control=p_control,
    delta=0.005,
    k=3,
)

trial = BinomialABTemplate(ledger)
protocol = BinomialABProtocol(**protocol.model_dump())
trial.set_protocol(protocol)

print(
    f"Design Initialized: n_max={int(protocol.method.stopping_policy.timer.max_sample_size)}"
)

In [ ]:
for i, (_, row) in enumerate(df.iterrows()):
    batch = []
    if row["dn_c"] > 0:
        batch.append(ArmData(n=row["dn_c"], success=row["ds_c"], arm="C"))
    if row["dn_t"] > 0:
        batch.append(ArmData(n=row["dn_t"], success=row["ds_t"], arm="T"))

    # update() returns minimal status info
    trial.update(batch)
    res = trial.report_progress()

    if res.get("look"):
        print(f"Look {res['look']} triggered. Status: {res['status']}")
        print(f"  Progress: {res}")

        if res["status"] == "STOP_EFFICACY":
            print("Stopping study early!")
            final = trial.report_result()
            print(f"Final Analysis: {final}")
            break

## 3. Historical Analysis: The `run_backtest` API

The `run_backtest` method remains all-in-one, returning a comprehensive `FinalReport` for ease of use.

In [ ]:
def batch_stream(df):
    for _, row in df.iterrows():
        batch = []
        if row["dn_c"] > 0:
            batch.append(ArmData(n=row["dn_c"], success=row["ds_c"], arm="C"))
        if row["dn_t"] > 0:
            batch.append(ArmData(n=row["dn_t"], success=row["ds_t"], arm="T"))
        yield batch


bt_ledger = Ledger(ibis.duckdb.connect(":memory:"), "backtest_events")
bt_ledger.ensure()
bt_trial = BinomialABTemplate(bt_ledger)
bt_trial.set_protocol(protocol)  # Using the already realized protocol

print("Starting historical backtest...")
bt_report = bt_trial.backtest(batch_stream(df))
print(f"Backtest Final Report: {bt_report}")